In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import joblib
from pytrends.request import TrendReq
from datetime import timedelta, time
from random import random
import time

In [ ]:
df= pd.read_excel("ML Project.xlsx")
print(df.head())

In [ ]:
# Convert release date
df['Release_Date'] = pd.to_datetime(df['Release_Date'], errors='coerce')
# Extract date features
df['Release_Year'] = df['Release_Date'].dt.year
df['Release_Month'] = df['Release_Date'].dt.month
df['Release_Quarter'] = df['Release_Date'].dt.quarter
# Clean and split genre/language columns
def clean_multi_value_column(x):
    # If it's already a list (e.g., ['Action', 'Drama'])
    if isinstance(x, list):
        return [str(i).strip() for i in x if str(i).strip()]
    # If it's a string (e.g., "Action, Drama")
    elif isinstance(x, str):
        return [i.strip() for i in x.split(',') if i.strip()]
    # Anything else (NaN, None, etc.)
    else:
        return []

# Apply the function safely
df['Genres'] = df['Genres'].apply(clean_multi_value_column)
df['Languages'] = df['Languages'].apply(clean_multi_value_column)

# Multi-hot encode genres and languages
mlb_genre = MultiLabelBinarizer()
mlb_lang = MultiLabelBinarizer()

genre_encoded = pd.DataFrame(mlb_genre.fit_transform(df['Genres']), columns=mlb_genre.classes_)
lang_encoded = pd.DataFrame(mlb_lang.fit_transform(df['Languages']), columns=mlb_lang.classes_)

# Combine everything
df = pd.concat([df, genre_encoded, lang_encoded], axis=1)

In [ ]:
print(df[['Genres', 'Languages']].head(10))

In [ ]:
# df['Genre_Trend_Score'] = df.apply(lambda row: get_genre_trend_score(row['Genres'], row['Release Date']), axis=1)

df['Genre_Trend_Score'] = df['Genre_Trend_Score'].fillna(0)

In [ ]:
# Define your target
y = df['Success/Failure']

# List of columns to drop — using actual names from your dataframe
drop_cols = ['Movie_Title', 'Release_Date', 'Genres', 'Languages', 'Success/Failure']

# Drop only the ones that exist in your dataframe (avoids KeyError)
X = df.drop(columns=[col for col in drop_cols if col in df.columns])


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=300, random_state=42)
rf_model.fit(X_train_scaled, y_train)

y_pred = rf_model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


In [ ]:
print(y.value_counts(normalize=True))
## since data is highly imbalanced we are using SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
X = df.drop(columns=[col for col in ['Movie_Title', 'Release_Date', 'Genres', 'Languages', 'Success/Failure'] if col in df.columns])
y = df['Success/Failure']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Before SMOTE Training data class balance:\n{y_train.value_counts(normalize=True)}")

# Handle imbalance with SMOTE
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f"\nAfter SMOTE Training data class balance:\n{y_train_res.value_counts(normalize=True)}")
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)

# Train model
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced_subsample',  # gives extra weight to minority class
    max_depth=None)
model.fit(X_train_scaled, y_train_res)
# Evaluate
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

print("\nModel Evaluation (After Balancing):\n")
print(classification_report(y_test, y_pred))
print("ROC-AUC Score:", round(roc_auc_score(y_test, y_pred_proba), 3))

#Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
## even after SMOT, model is still predicting all "Failure"
# using different approach

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, recall_score
from sklearn.model_selection import GridSearchCV

log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train_res)
y_pred_lr = log_reg.predict(X_test_scaled)
y_pred_proba_lr = log_reg.predict_proba(X_test_scaled)[:, 1]
print(classification_report(y_test, y_pred_lr))
print("ROC-AUC:", round(roc_auc_score(y_test, y_pred_proba_lr), 3))

In [ ]:
# The scale_pos_weight parameter makes XGBoost treat successes as equally important despite being rare.Handles imbalance better
xgb = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y_train.value_counts()[0] / y_train.value_counts()[1]),
    random_state=42
)

xgb.fit(X_train_scaled, y_train_res)
y_pred_xgb = xgb.predict(X_test_scaled)
y_proba_xgb = xgb.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba_xgb), 3))

In [ ]:
#prioritize detecting “success” (class 1)
param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [4, 6, 8]}
grid = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_grid,
    scoring=make_scorer(recall_score, pos_label=1),
    cv=5)
grid.fit(X_train_scaled, y_train_res)
print("Best params:", grid.best_params_)

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances.head(15))

plt.figure(figsize=(8, 5))
importances.head(15).plot(kind='bar')
plt.title('Top 15 Feature Importances')
plt.show()
## Even though Genre_Trend_Score dominates, the model may depend too heavily on it,
## suggesting the other features are underpowered or too noisy.

In [ ]:
# --- Imports ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix, accuracy_score, f1_score)
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare Data ---
y = df['Success/Failure']
X = df.drop(columns=[col for col in ['Movie_Title', 'Release_Date', 'Genres', 'Languages', 'Success/Failure'] if col in df.columns])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Handle imbalance with SMOTE ---
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts(normalize=True))
print("After SMOTE:", y_train_res.value_counts(normalize=True))

# Scale numeric features ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)

# Initialize Models ---
models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=300, random_state=42, class_weight='balanced_subsample'
    ),
    "XGBoost": XGBClassifier(
        n_estimators=400, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(y_train.value_counts()[0] / y_train.value_counts()[1]),
        random_state=42, use_label_encoder=False, eval_metric='logloss'
    ),
    "LogisticRegression": LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=42)}

# Train & Evaluate ---
results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train_res)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results.append({
        "Model": name,
        "Accuracy": acc,
        "F1": f1,
        "ROC_AUC": auc})

    print(f"\n{'='*50}")
    print(f"Results for {name}")
    print(classification_report(y_test, y_pred))
    print("ROC-AUC:", round(auc, 3))
    print(confusion_matrix(y_test, y_pred))

# Compare Model Results ---
results_df = pd.DataFrame(results)
print("\n Model Comparison Summary:\n")
print(results_df)

# Visualization ---
plt.figure(figsize=(8,5))
sns.barplot(data=results_df.melt(id_vars='Model', value_vars=['Accuracy', 'F1', 'ROC_AUC']),
            x='Model', y='value', hue='variable')
plt.title('Model Comparison')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.legend(title='Metric')
plt.show()